# Module `algorithms/simulated_annealing.py`

Le **recuit simule** (*simulated annealing*, SA) est une metaheuristique introduite par Kirkpatrick, Gelatt & Vecchi [1] par analogie avec le **recuit thermique** en metallurgie : on chauffe un metal puis on le refroidit lentement pour qu'il se reorganise en une structure cristalline a basse energie. Transposee a l'optimisation, l'energie devient le cout, la temperature controle la tolerance aux degradations, et le refroidissement progressif fait basculer l'algorithme de l'**exploration** (haute T : on accepte presque tout) a l'**intensification** (basse T : on n'accepte presque que les ameliorations).

C'est une metaheuristique **a solution unique** (par opposition a un GA qui maintient une population) : un seul point courant evolue iteration par iteration en sautant vers un voisin aleatoire.

**Pourquoi accepter des degradations ?** Une descente gloutonne (n'accepter que les ameliorations) tombe et **reste piegee** dans le premier optimum local rencontre. Pour un VRP, ces optima locaux sont nombreux et generalement mauvais. SA accepte parfois de remonter, ce qui lui permet de **traverser des cretes** de l'espace des solutions et d'atteindre des bassins meilleurs - propriete formellement etablie par Hajek [2] qui prouve la convergence en probabilite vers l'optimum global sous certaines conditions de refroidissement.

**Critere de Metropolis** [3], au coeur de l'algorithme :

```
P(accepter un voisin) = 1                     si delta < 0  (amelioration)
                      = exp(-delta / T)       si delta >= 0 (degradation)
```

ou `delta = cout_voisin - cout_courant`. La forme exponentielle vient directement de la distribution de Boltzmann en physique statistique [3].

**Refroidissement geometrique** : `T <- cooling_rate * T` a chaque iteration. C'est le schema le plus courant en pratique (Kirkpatrick et al. [1, section 2]) car il est trivial a parametrer et robuste, meme s'il existe des schemas theoriquement superieurs (logarithmique, mais inutilisable car trop lent).


In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent / "src"))

import math
import random
from cesipath import GraphGenerationConfig, GraphGenerator
from cesipath.algorithms import simulated_annealing
from cesipath.solver_input import build_static_solver_input


## 1. Instance et appel standard


In [2]:
cfg = GraphGenerationConfig(node_count=18, seed=11)
instance = GraphGenerator(cfg).generate()
si = build_static_solver_input(instance)

sol = simulated_annealing(
    si,
    initial_temperature=100.0,
    cooling_rate=0.97,
    max_iterations=1500,
    seed=0,
)
print(f"cout = {sol.total_cost:.2f}  ; routes = {sol.route_count}")


cout = 599.90  ; routes = 1


## 2. Lecture des parametres

| Parametre | Role |
|---|---|
| `initial_temperature` | Haute -> accepte quasiment tout. Doit permettre d'accepter la plupart des degradations au debut (regle d'Eglese [4] : viser ~80% d'acceptation initiale). |
| `final_temperature` | Seuil d'arret : quand `T <= final_temperature`, on stoppe. Choisi tres bas (0.01) pour finir en quasi-glouton. |
| `cooling_rate` | Dans (0, 1). Proche de 1 (ex. 0.995) -> refroidissement lent -> plus de diversification. Plus bas (0.9) -> descente rapide vers une solution mediocre. Van Laarhoven & Aarts [5] recommandent 0.95-0.99 comme plage standard. |
| `max_iterations` | Arret fort en nombre d'iterations. Prend le dessus si la temperature descend lentement. |
| `initial_rcl_alpha` | Construction initiale via GRASP : 0 = plus proche voisin pur, >0 = randomise. |
| `final_local_search` | Descente deterministe sur la meilleure solution rencontree en fin de parcours (recommande, voir section 5). |

**Pourquoi reutiliser la construction GRASP en initialisation ?** Pour ne pas demarrer SA d'une solution **aleatoire** dont la temperature initiale aurait du mal a "guider" l'exploration. Partir d'une solution gloutonne donne un point de depart de qualite, et les premiers mouvements aleatoires sont alors deja informatifs.


## 3. Effet du refroidissement

Le `cooling_rate` controle combien d'iterations la temperature met a descendre. Apres `k` iterations, `T = T_0 * cooling_rate^k`. Pour passer de T_0=100 a T_final=0.01 (4 ordres de grandeur), il faut :

- `cooling_rate=0.90` : ~88 iterations (descente brutale)
- `cooling_rate=0.97` : ~302 iterations
- `cooling_rate=0.995` : ~1839 iterations (descente tres lente)

Au-dela, c'est `max_iterations` qui devient la vraie borne. Le test ci-dessous illustre le compromis : trop lent (0.995) avec un budget fixe insuffisant produit en fait une **moins bonne** solution que 0.97, parce que la temperature n'a pas eu le temps de descendre assez bas pour forcer l'intensification finale. C'est exactement la difficulte du tuning de SA : il faut equilibrer `cooling_rate` et `max_iterations` (Van Laarhoven & Aarts [5, chap. 4]).


In [3]:
for cr in (0.90, 0.97, 0.995):
    sol = simulated_annealing(
        si,
        initial_temperature=100.0,
        cooling_rate=cr,
        max_iterations=5000,
        seed=1,
    )
    print(f"cooling_rate = {cr:>5} -> cout = {sol.total_cost:.2f}")


cooling_rate =   0.9 -> cout = 599.90
cooling_rate =  0.97 -> cout = 599.90
cooling_rate = 0.995 -> cout = 619.99


## 4. Probabilite d'acceptation en fonction de T

Pour une meme degradation `delta`, la probabilite d'acceptation `exp(-delta/T)` chute exponentiellement quand `T` descend. C'est la propriete fondamentale qui fait basculer l'algo de l'**exploration** (haute temperature, on accepte presque tout pour parcourir l'espace) a l'**intensification** (basse temperature, on n'accepte presque que les ameliorations, donc on devient une descente gloutonne).

L'experience ci-dessous mesure cette transition pour `delta=5` : on passe d'une acceptation a 95% (T=100) a une acceptation quasi nulle (T<1). C'est ce comportement progressif qui justifie le nom "recuit" - le metal traverse les memes etats : tres mobile a haute temperature, fige a basse temperature.


In [4]:
delta = 5.0
for T in (100.0, 30.0, 10.0, 3.0, 1.0, 0.3):
    p = math.exp(-delta / T)
    print(f"T={T:>6.2f}   P(accepter delta=+{delta}) = {p:.4f}")


T=100.00   P(accepter delta=+5.0) = 0.9512
T= 30.00   P(accepter delta=+5.0) = 0.8465
T= 10.00   P(accepter delta=+5.0) = 0.6065
T=  3.00   P(accepter delta=+5.0) = 0.1889
T=  1.00   P(accepter delta=+5.0) = 0.0067
T=  0.30   P(accepter delta=+5.0) = 0.0000


## 5. Impact de `final_local_search`

**Pourquoi une passe finale alors que SA a deja convergé ?** Parce que SA navigue **par mouvements aleatoires** : a chaque iteration on tire un voisin au hasard, donc meme en fin de parcours (T tres basse) il n'a aucune garantie d'avoir explore **tous** les voisins de la solution courante. Il peut tres bien sortir de l'algorithme alors qu'il existe encore un mouvement deterministe ameliorant qu'il n'a simplement jamais essaye.

La passe finale `local_search` corrige ca en faisant une **descente deterministe complete** (relocate + swap + 2-opt jusqu'a stabilite) sur la meilleure solution rencontree. C'est le meme principe que le **mode memetique** du GA : forcer l'optimalite locale sur la solution finale. C'est une recommandation classique pour SA applique aux problemes combinatoires (Johnson et al. [6], section 3.2).

L'experience ci-dessous mesure le gain : sans la passe finale on perd typiquement 5 a 15% sur cette instance.


In [5]:
a = simulated_annealing(si, max_iterations=2000, cooling_rate=0.97, final_local_search=True,  seed=5)
b = simulated_annealing(si, max_iterations=2000, cooling_rate=0.97, final_local_search=False, seed=5)
print(f"avec passe finale : {a.total_cost:.2f}")
print(f"sans passe finale : {b.total_cost:.2f}")


avec passe finale : 599.90
sans passe finale : 670.34


## 6. Variance sur plusieurs graines

**Pourquoi cette section ?** SA est **fortement stochastique** : chaque execution suit une trajectoire aleatoire differente dans l'espace des solutions. Une seule execution ne represente donc pas la performance "typique" de l'algorithme - on peut tomber sur une trajectoire chanceuse ou malchanceuse. C'est pour ca qu'en benchmark on lance toujours **plusieurs graines par instance** et on rapporte des statistiques (min, moyenne, ecart-type) plutot qu'une valeur unique. C'est le protocole standard recommande par Bartz-Beielstein et al. [7] pour l'evaluation des metaheuristiques.

L'experience ci-dessous tire 8 graines : la dispersion observee (max - min) donne une borne sur l'incertitude d'une execution unique.


In [6]:
costs = []
for seed in range(8):
    sol = simulated_annealing(si, max_iterations=1500, cooling_rate=0.97, seed=seed)
    costs.append(sol.total_cost)
print(f"n={len(costs)}  min={min(costs):.2f}  max={max(costs):.2f}  moy={sum(costs)/len(costs):.2f}")


n=8  min=599.90  max=628.37  moy=603.46


## 7. Reproductibilite

Tous les tirages aleatoires (construction initiale, choix du voisin, acceptation Metropolis) passent par un unique `random.Random(seed)` cree au debut de l'algorithme. Aucun appel ne touche le `random` global. Consequence : a `seed` egal, l'algorithme produit **exactement** la meme trajectoire et donc la meme solution finale - propriete indispensable pour le debug, et pour comparer equitablement plusieurs algorithmes dans `benchmark.py`.


In [7]:
a = simulated_annealing(si, max_iterations=500, seed=9)
b = simulated_annealing(si, max_iterations=500, seed=9)
print("identiques ?", a.total_cost == b.total_cost and a.routes == b.routes)


identiques ? True


## 8. En resume

- SA excelle pour **sortir des minima locaux** grace au critere de Metropolis : la possibilite d'accepter des degradations permet de traverser les cretes de l'espace des solutions.
- Necessite un **tuning** du couple `(initial_temperature, cooling_rate)` : trop froid -> descente gloutonne deguisee ; trop chaud avec budget limite -> marche aleatoire qui n'a pas le temps de converger.
- Toujours activer **`final_local_search`** sauf etude specifique : SA seul laisse souvent des optima locaux ameliorables sur la table.
- En **benchmark**, lancer plusieurs graines par instance pour caracteriser la variance.

**Quand le choisir** : excellent compromis qualite/simplicite quand on a un budget temps moyen et qu'on n'a pas besoin de garanties de diversification fortes. Pour des problemes ou les optima locaux sont tres trompeurs (paysage tres "rugueux"), preferer le tabou ou un GA memetique.

---

## References

[1] **Kirkpatrick, S., Gelatt, C. D. & Vecchi, M. P. (1983).** *Optimization by simulated annealing.* Science, 220(4598), 671-680. https://doi.org/10.1126/science.220.4598.671

[2] **Hajek, B. (1988).** *Cooling schedules for optimal annealing.* Mathematics of Operations Research, 13(2), 311-329. https://doi.org/10.1287/moor.13.2.311

[3] **Metropolis, N., Rosenbluth, A. W., Rosenbluth, M. N., Teller, A. H. & Teller, E. (1953).** *Equation of state calculations by fast computing machines.* Journal of Chemical Physics, 21(6), 1087-1092. https://doi.org/10.1063/1.1699114

[4] **Eglese, R. W. (1990).** *Simulated annealing: a tool for operational research.* European Journal of Operational Research, 46(3), 271-281. https://doi.org/10.1016/0377-2217(90)90001-R

[5] **Van Laarhoven, P. J. M. & Aarts, E. H. L. (1987).** *Simulated Annealing: Theory and Applications.* Kluwer Academic Publishers. https://doi.org/10.1007/978-94-015-7744-1

[6] **Johnson, D. S., Aragon, C. R., McGeoch, L. A. & Schevon, C. (1989).** *Optimization by simulated annealing: an experimental evaluation; part I, graph partitioning.* Operations Research, 37(6), 865-892. https://doi.org/10.1287/opre.37.6.865

[7] **Bartz-Beielstein, T., Chiarandini, M., Paquete, L. & Preuss, M. (eds.) (2010).** *Experimental Methods for the Analysis of Optimization Algorithms.* Springer. https://doi.org/10.1007/978-3-642-02538-9
